Transformer Demo


In [1]:
import torch
import torch.nn as nn
import math
import time

# 模型配置参数
config = {
    "vocab_size": 50257,   # 词汇表大小，参考GPT-2
    "seq_len": 1024,       # 输入序列长度
    "batch_size": 8,       # 批处理大小
    "d_model": 768,        # 隐藏层维度/词嵌入维度
    "nhead": 12,           # 多头注意力机制中的头数
    "num_layers": 12,      # Transformer编码器的层数
    "dim_feedforward": 3072, # 前馈神经网络的隐藏层维度
    "device": "cpu"        # 初始设备
}

# 位置编码层：为输入序列添加位置信息
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)  # 将位置编码注册为缓冲区，不参与梯度更新

    def forward(self, x):
        x = x + self.pe[:x.size(1)]  # 将位置编码加到输入上
        return x

# Transformer模型
class TransformerModel(nn.Module):
    def __init__(self, config):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.pos_encoder = PositionalEncoding(config["d_model"])
        # 定义Transformer编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            batch_first=True  # 输入输出维度为 (batch, seq, feature)
        )
        # 堆叠多个编码器层
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config["num_layers"])
        self.fc = nn.Linear(config["d_model"], config["vocab_size"])  # 输出线性层，预测下一个词的概率

    def forward(self, src):
        src = self.embedding(src) * math.sqrt(config["d_model"])  # 词嵌入并缩放
        src = self.pos_encoder(src)  # 添加位置编码
        output = self.transformer(src)  # 通过Transformer编码器
        return self.fc(output)  # 线性变换输出

# CPU版本推理函数
def cpu_version():
    config["device"] = "cpu"
    torch.set_num_threads(8)  # 设置CPU线程数以加速计算
    model = TransformerModel(config).to(config["device"])
    model.eval()  # 设置为评估模式
    # 生成随机输入数据（模拟一个批次的数据）
    inputs = torch.randint(0, config["vocab_size"], (config["batch_size"], config["seq_len"])).to(config["device"])
    
    # 预热：第一次推理可能较慢，先运行一次
    with torch.no_grad():
        result = model(inputs)
        print(result)
    
    # 正式计时推理
    start_time = time.time()
    with torch.no_grad():
        result2 = model(inputs)
        print(result2)
    elapsed = time.time() - start_time
    print(f"CPU推理时间: {elapsed:.4f}秒")

# GPU版本推理函数（如果可用）
def gpu_version():
    if not torch.cuda.is_available():
        print("CUDA不可用，无法运行GPU版本")
        return
    config["device"] = "cuda"
    torch.backends.cudnn.benchmark = True  # 启用cuDNN基准优化，加速计算
    model = TransformerModel(config).to(config["device"])
    model.eval()
    inputs = torch.randint(0, config["vocab_size"], (config["batch_size"], config["seq_len"])).to(config["device"])
    
    # GPU预热：运行几次以确保CUDA初始化完成
    with torch.no_grad():
        for _ in range(3):
            _ = model(inputs)
    
    # 使用CUDA事件进行精确计时（GPU时间）
    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)
    starter.record()
    with torch.no_grad():
        _ = model(inputs)
    ender.record()
    torch.cuda.synchronize()  # 等待CUDA操作完成
    elapsed = starter.elapsed_time(ender) / 1000  # 将毫秒转换为秒
    print(f"GPU推理时间: {elapsed:.4f}秒")

# 主程序入口
if __name__ == "__main__":
    print("运行CPU版本:")
    cpu_version()
    print("\n运行GPU版本:")
    gpu_version()

运行CPU版本:
tensor([[[ 2.1093e-01, -7.9863e-01, -2.1259e-01,  ...,  6.6395e-01,
           1.0156e-01, -4.3468e-01],
         [ 6.2662e-02, -4.1110e-01, -5.9290e-01,  ...,  1.1484e+00,
           1.9070e-01, -5.6005e-01],
         [-1.6785e-02, -5.1439e-01, -3.4212e-01,  ...,  7.9311e-01,
           2.3858e-01, -5.1847e-01],
         ...,
         [ 1.0032e-01, -6.7671e-01, -3.3164e-01,  ...,  5.0012e-01,
           1.8713e-01, -4.6600e-01],
         [ 4.3624e-02, -4.1535e-01, -4.4730e-01,  ...,  8.2179e-01,
           2.4154e-01, -2.5202e-01],
         [ 1.0998e-01, -6.3205e-01, -3.4828e-01,  ...,  7.6217e-01,
           3.7733e-01, -3.7607e-01]],

        [[ 3.3135e-01, -4.6450e-01, -4.3578e-01,  ...,  6.6845e-01,
           4.1007e-01, -4.9604e-01],
         [ 3.0052e-01, -6.2977e-01, -1.4201e-01,  ...,  7.8094e-01,
           5.0039e-01, -4.3066e-01],
         [ 1.8338e-01, -7.4969e-01, -5.2702e-01,  ...,  6.2486e-01,
          -2.8630e-02, -3.7328e-01],
         ...,
         [-1.853

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import time

# 模型配置参数
config = {
    "vocab_size": 10000,    # 词汇表大小
    "seq_len": 64,          # 输入序列长度
    "batch_size": 32,       # 批处理大小
    "d_model": 512,         # 隐藏层维度/词嵌入维度
    "nhead": 8,             # 多头注意力机制中的头数
    "num_layers": 6,        # Transformer编码器的层数
    "dim_feedforward": 2048, # 前馈神经网络的隐藏层维度
    "dropout": 0.1,         # Dropout率
    "lr": 0.0001,           # 学习率
    "num_epochs": 10,       # 训练轮数
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# 位置编码层：为输入序列添加位置信息
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# Transformer语言模型
class TransformerLanguageModel(nn.Module):
    def __init__(self, config):
        super(TransformerLanguageModel, self).__init__()
        self.embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.pos_encoder = PositionalEncoding(config["d_model"])
        self.dropout = nn.Dropout(config["dropout"])
        
        # Transformer编码器层
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config["num_layers"])
        self.fc = nn.Linear(config["d_model"], config["vocab_size"])
        
    def forward(self, src, src_mask=None):
        src = self.embedding(src) * math.sqrt(config["d_model"])
        src = self.pos_encoder(src)
        src = self.dropout(src)
        
        if src_mask is None:
            src_mask = self.generate_square_subsequent_mask(src.size(1)).to(src.device)
        
        output = self.transformer(src, src_mask)
        return self.fc(output)
    
    def generate_square_subsequent_mask(self, sz):
        """生成掩码，防止模型看到未来信息"""
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

# 生成随机训练数据
def generate_random_data(config, num_samples=1000):
    # 在实际应用中，这里应该使用真实的数据集
    data = torch.randint(0, config["vocab_size"], (num_samples, config["seq_len"]))
    # 对于语言模型，输入是序列的前n-1个词，目标是后n-1个词
    inputs = data[:, :-1]
    targets = data[:, 1:]
    return inputs, targets

# 训练函数
def train_model(model, train_loader, criterion, optimizer, config):
    model.train()
    total_loss = 0
    
    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(config["device"]), targets.to(config["device"])
        
        optimizer.zero_grad()
        
        # 生成掩码
        src_mask = model.generate_square_subsequent_mask(data.size(1)).to(config["device"])
        
        # 前向传播
        output = model(data, src_mask)
        
        # 计算损失 - 只计算最后一个时间步的损失
        loss = criterion(output.view(-1, config["vocab_size"]), targets.contiguous().view(-1))
        
        # 反向传播和优化
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪
        optimizer.step()
        
        total_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')
    
    return total_loss / len(train_loader)

# 预测下一个词
def predict_next_word(model, input_sequence, config, top_k=5):
    model.eval()
    with torch.no_grad():
        # 生成掩码
        src_mask = model.generate_square_subsequent_mask(input_sequence.size(1)).to(config["device"])
        
        # 获取模型输出
        output = model(input_sequence, src_mask)
        
        # 获取最后一个时间步的输出
        last_time_step_output = output[:, -1, :]
        
        # 应用softmax获取概率分布
        probabilities = torch.softmax(last_time_step_output, dim=-1)
        
        # 获取top-k最可能的词和概率
        top_k_probs, top_k_indices = torch.topk(probabilities, top_k, dim=-1)
        
        return top_k_probs.cpu().numpy(), top_k_indices.cpu().numpy()

# 主函数
def main():
    print(f"使用设备: {config['device']}")
    
    # 初始化模型
    model = TransformerLanguageModel(config).to(config["device"])
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
    
    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    # 生成训练数据
    print("生成训练数据...")
    inputs, targets = generate_random_data(config, num_samples=1000)
    print(inputs)
    print(targets)
    
    # 创建DataLoader
    dataset = torch.utils.data.TensorDataset(inputs, targets)
    train_loader = torch.utils.data.DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=True
    )
    
    # 训练模型
    print("开始训练...")
    for epoch in range(config["num_epochs"]):
        start_time = time.time()
        avg_loss = train_model(model, train_loader, criterion, optimizer, config)
        epoch_time = time.time() - start_time
        
        print(f'Epoch {epoch+1}/{config["num_epochs"]}, '
              f'平均损失: {avg_loss:.4f}, '
              f'时间: {epoch_time:.2f}秒')
    
    # 演示下一个词预测
    print("\n演示下一个词预测:")
    test_input = torch.randint(0, config["vocab_size"], (1, 10)).to(config["device"])
    print(f"输入序列: {test_input.cpu().numpy()}")
    
    probs, indices = predict_next_word(model, test_input, config, top_k=3)
    print(f"最可能的下一个词: {indices[0]}")
    print(f"对应概率: {probs[0]}")
    
    # 保存模型
    torch.save(model.state_dict(), 'transformer_lm.pth')
    print("模型已保存到 transformer_lm.pth")

if __name__ == "__main__":
    main()

load & predict

In [8]:
import torch
import torch.nn as nn
import math

# 假设的配置参数 (必须与训练时完全相同!)
config = {
    "vocab_size": 10000,    # 词汇表大小
    "seq_len": 64,          # 输入序列长度 (注意：在预测时，输入序列长度可以小于或等于此值，但不应超过训练时模型见过的最大长度)
    "d_model": 512,         # 隐藏层维度/词嵌入维度
    "nhead": 8,             # 多头注意力机制中的头数
    "num_layers": 6,        # Transformer编码器的层数
    "dim_feedforward": 2048, # 前馈神经网络的隐藏层维度
    "dropout": 0.1,         # Dropout率
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# 1. 定义模型结构 (必须与训练时完全一致!)
class PositionalEncoding(nn.Module):
    # ... (与训练代码中完全相同的定义)
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerLanguageModel(nn.Module):
    # ... (与训练代码中完全相同的定义)
    def __init__(self, config):
        super(TransformerLanguageModel, self).__init__()
        self.embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.pos_encoder = PositionalEncoding(config["d_model"])
        self.dropout = nn.Dropout(config["dropout"])
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config["d_model"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=config["num_layers"])
        self.fc = nn.Linear(config["d_model"], config["vocab_size"])
        
    def forward(self, src, src_mask=None):
        src = self.embedding(src) * math.sqrt(config["d_model"])
        src = self.pos_encoder(src)
        src = self.dropout(src)
        
        if src_mask is None:
            src_mask = self.generate_square_subsequent_mask(src.size(1)).to(src.device)
        
        output = self.transformer(src, src_mask)
        return self.fc(output)
    
    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

# 2. 实例化模型并加载权重
print("加载训练好的模型...")
model = TransformerLanguageModel(config).to(config["device"]) # 创建模型实例
model.load_state_dict(torch.load('transformer_lm.pth', map_location=config["device"])) # 加载权重
model.eval() # 将模型设置为评估模式[5](@ref)
print("模型加载完成!")

# 3. 定义预测函数 (与训练代码中的predict_next_word函数类似)
def predict_next_word(model, input_sequence, top_k=5):
    """
    使用加载的模型预测下一个词
    :param model: 加载的模型
    :param input_sequence: 输入序列，形状为 [batch_size, seq_len]
    :param top_k: 返回最可能的前k个词
    :return: 最可能的top_k个词的概率和索引
    """
    model.eval() # 确保模型在评估模式
    with torch.no_grad(): # 禁用梯度计算，节省内存和计算资源[5](@ref)
        # 生成掩码
        src_mask = model.generate_square_subsequent_mask(input_sequence.size(1)).to(config["device"])
        
        # 获取模型输出
        output = model(input_sequence, src_mask)
        
        # 获取最后一个时间步的输出
        last_time_step_output = output[:, -1, :]
        
        # 应用softmax获取概率分布
        probabilities = torch.softmax(last_time_step_output, dim=-1)
        
        # 获取top-k最可能的词和概率
        top_k_probs, top_k_indices = torch.topk(probabilities, top_k, dim=-1)
        
        return top_k_probs.cpu().numpy(), top_k_indices.cpu().numpy()

# 4. 准备输入数据并进行预测
# 注意：在实际应用中，你的输入数据可能需要经过与训练数据相同的预处理（如分词、映射到ID等）
# 这里我们创建一个模拟的输入序列（假设是token ID的序列）
# 假设我们的输入序列长度为10，词汇表大小与训练时一致（0到9999）
input_sequence = torch.randint(0, config["vocab_size"], (1, 10)).to(config["device"]) # 形状 [1, 10] (batch_size=1, seq_len=10)
print(f"输入序列 (token IDs): {input_sequence.cpu().numpy()}")

# 5. 执行预测
probs, indices = predict_next_word(model, input_sequence, top_k=3)
print(f"最可能的下一个词 (Token IDs): {indices[0]}")
print(f"对应的概率: {probs[0]}")

# 你可以进一步将预测出的token ID映射回实际的词汇（如果你有id2word的字典）
# predicted_word = id2word[indices[0][0]]
# print(f"预测的下一个词是: {predicted_word}")

加载训练好的模型...
模型加载完成!
输入序列 (token IDs): [[1935  802 9014 9351 7321 1376 2773 3871 2475 3634]]
最可能的下一个词 (Token IDs): [4285 3539 5123]
对应的概率: [0.00077352 0.00075489 0.00075251]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# 1. 准备示例数据和构建词汇表 (Vocabulary)
data = [
    ("你好", "你好！有什么我可以帮助你的,SB？"),
    ("今天天气怎么样？", "今天天气很好，阳光明媚。"),
    ("你会做什么？", "我可以和你聊天，回答你的问题。"),
    ("你是傻逼吗？", "是的,你也是。")
]

# 构建词汇表，包含所有字符和特殊标记
vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}  # 特殊标记
for pair in data:
    for sentence in pair:
        for char in sentence:
            if char not in vocab:
                vocab[char] = len(vocab)
# 反向词汇表，用于从ID转换回字符
id_to_char = {id: char for char, id in vocab.items()}

vocab_size = len(vocab)  # 词汇表大小

# 编码函数：将字符串转换为ID列表，并添加起止符
def encode(sentence, vocab):
    return [vocab["<SOS>"]] + [vocab[char] for char in sentence] + [vocab["<EOS>"]]

# 解码函数：将ID列表转换回字符串，并去除起止符
def decode(id_list, id_to_char):
    return ''.join([id_to_char[id] for id in id_list if id not in [vocab["<SOS>"], vocab["<EOS>"], vocab["<PAD>"]]])

# 对数据进行编码和填充，确保长度一致
max_len = 20
def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

encoded_data = []
for pair in data:
    src_encoded = encode(pair[0], vocab)
    tgt_encoded = encode(pair[1], vocab)
    # 对输入和目标分别进行填充
    src_padded = pad_sequence(src_encoded, max_len, vocab["<PAD>"])
    tgt_padded = pad_sequence(tgt_encoded, max_len, vocab["<PAD>"])
    encoded_data.append((src_padded, tgt_padded))

# 转换为Tensor
src_data = torch.tensor([pair[0] for pair in encoded_data], dtype=torch.long)
tgt_data = torch.tensor([pair[1] for pair in encoded_data], dtype=torch.long)

# 2. 定义模型
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe) 

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, src):
        src_emb = self.embedding(src)
        src_emb = self.pos_encoder(src_emb)
        output = self.transformer_encoder(src_emb)
        output = self.fc(output)
        return output

# 超参数
d_model = 16
nhead = 2
num_layers = 2
max_len = 20

model = SimpleTransformer(vocab_size, d_model, nhead, num_layers, max_len)

# 3. 训练模型
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"]) # 忽略填充位的损失
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(src_data) # 前向传播
    # 计算损失时，将输出展平，目标移一位（用于下一个词预测）
    loss = criterion(output.view(-1, vocab_size), tgt_data.contiguous().view(-1))
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# 4. 保存模型
torch.save(model.state_dict(), "transformer_chatbot.pth")
print("模型已保存为 'transformer_chatbot.pth'")

# 5. 加载模型（模拟在另一个脚本中加载）
# 首先，需要重新实例化模型结构
loaded_model = SimpleTransformer(vocab_size, d_model, nhead, num_layers, max_len)
loaded_model.load_state_dict(torch.load("transformer_chatbot.pth"))
loaded_model.eval() # 设置为评估模式
print("模型加载成功！")

# 6. 文本生成函数（输入输出均为文字）
def generate_reply(model, input_sentence, vocab, id_to_char, max_length=20):
    model.eval()
    # 将输入文字编码为ID序列，并进行填充
    input_encoded = encode(input_sentence, vocab)
    input_padded = pad_sequence(input_encoded, max_len, vocab["<PAD>"])
    input_tensor = torch.tensor([input_padded], dtype=torch.long)

    with torch.no_grad():
        output = model(input_tensor)
        # 获取预测的ID（取每个位置概率最大的词）
        predicted_ids = output.argmax(dim=-1).squeeze(0).tolist()
    # 将预测的ID序列解码回文字
    reply = decode(predicted_ids, id_to_char)
    return reply

# 测试一下！
test_input = "傻逼"
predicted_reply = generate_reply(loaded_model, test_input, vocab, id_to_char)
print(f"输入: {test_input}")
print(f"模型回复: {predicted_reply}")

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

# 1. 准备示例数据和构建词汇表 (Vocabulary)
data = [
    ("你好", "你好！有什么我可以帮助你的,SB？"),
    ("今天天气怎么样？", "今天天气很好，阳光明媚。"),
    ("你会做什么？", "我可以和你聊天，回答你的问题。"),
    ("你是傻逼吗？", "是的,你也是。")
]

# 构建词汇表，包含所有字符和特殊标记
vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}  # 特殊标记
for pair in data:
    for sentence in pair:
        for char in sentence:
            if char not in vocab:
                vocab[char] = len(vocab)
# 反向词汇表，用于从ID转换回字符
id_to_char = {id: char for char, id in vocab.items()}

vocab_size = len(vocab)  # 词汇表大小

# 编码函数：将字符串转换为ID列表，并添加起止符
def encode(sentence, vocab):
    return [vocab["<SOS>"]] + [vocab[char] for char in sentence] + [vocab["<EOS>"]]

# 解码函数：将ID列表转换回字符串，并去除起止符
def decode(id_list, id_to_char):
    return ''.join([id_to_char[id] for id in id_list if id not in [vocab["<SOS>"], vocab["<EOS>"], vocab["<PAD>"]]])

# 对数据进行编码和填充，确保长度一致
max_len = 20
def pad_sequence(seq, max_len, pad_value):
    return seq + [pad_value] * (max_len - len(seq))

encoded_data = []
for pair in data:
    src_encoded = encode(pair[0], vocab)
    tgt_encoded = encode(pair[1], vocab)
    # 对输入和目标分别进行填充
    src_padded = pad_sequence(src_encoded, max_len, vocab["<PAD>"])
    tgt_padded = pad_sequence(tgt_encoded, max_len, vocab["<PAD>"])
    encoded_data.append((src_padded, tgt_padded))

# 转换为Tensor
src_data = torch.tensor([pair[0] for pair in encoded_data], dtype=torch.long)
tgt_data = torch.tensor([pair[1] for pair in encoded_data], dtype=torch.long)

# 2. 定义模型
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len):
        super(SimpleTransformer, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, src):
        src_emb = self.embedding(src)
        src_emb = self.pos_encoder(src_emb)
        output = self.transformer_encoder(src_emb)
        output = self.fc(output)
        return output

# 超参数
d_model = 16
nhead = 2
num_layers = 2
max_len = 20

model = SimpleTransformer(vocab_size, d_model, nhead, num_layers, max_len)


# 5. 加载模型（模拟在另一个脚本中加载）
# 首先，需要重新实例化模型结构
loaded_model = SimpleTransformer(vocab_size, d_model, nhead, num_layers, max_len)
loaded_model.load_state_dict(torch.load("transformer_chatbot.pth"))
loaded_model.eval() # 设置为评估模式
print("模型加载成功！")

# 6. 文本生成函数（输入输出均为文字）
def generate_reply(model, input_sentence, vocab, id_to_char, max_length=20):
    model.eval()
    # 将输入文字编码为ID序列，并进行填充
    input_encoded = encode(input_sentence, vocab)
    input_padded = pad_sequence(input_encoded, max_len, vocab["<PAD>"])
    input_tensor = torch.tensor([input_padded], dtype=torch.long)

    with torch.no_grad():
        output = model(input_tensor)
        # 获取预测的ID（取每个位置概率最大的词）
        predicted_ids = output.argmax(dim=-1).squeeze(0).tolist()
    # 将预测的ID序列解码回文字
    reply = decode(predicted_ids, id_to_char)
    return reply

# 测试一下！
test_input = "SBB"
predicted_reply = generate_reply(loaded_model, test_input, vocab, id_to_char)
print(f"输入: {test_input}")
print(f"模型回复: {predicted_reply}")

模型加载成功！
输入: SBB
模型回复: 你你你！很,,明明媚。,SSB


RAG

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

# --- 第1步：准备知识库 ---
# 假设我们有一些关于太阳系的文本数据
documents = [
    "太阳系是一个由太阳和围绕它运行的天体组成的系统。",
    "地球是太阳系中唯一已知存在生命的行星。",
    "火星是太阳系中的第四颗行星，俗称“红色星球”。",
    "木星是太阳系中最大的一颗行星。",
    "土星以其壮观的光环系统而闻名。"
]

# --- 第2步：检索阶段 (Retrieval) ---

print("--- 检索阶段 ---")

# 加载一个用于向量化的深度学习模型
# all-MiniLM-L6-v2 是一个轻量且效果不错的模型
retrieval_model = SentenceTransformer('all-MiniLM-L6-v2')

# 将知识库中的文档转换为向量
doc_embeddings = retrieval_model.encode(documents, convert_to_tensor=True)

# 使用 Faiss 构建一个向量索引
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings.cpu().numpy())

# 用户提问
question = "太阳系中哪颗行星以其光环著称？"
print(f"用户问题: {question}")

# 将用户问题转换为向量
question_embedding = retrieval_model.encode(question, convert_to_tensor=True)

# 在向量索引中搜索最相似的文档
# k=1 表示我们只找最相似的一个
D, I = index.search(question_embedding.cpu().numpy().reshape(1, -1), k=1)

# 获取最相似文档的索引和内容
retrieved_doc_index = I[0][0]
retrieved_context = documents[retrieved_doc_index]

print(f"检索到的上下文: {retrieved_context}")

# --- 第3步：生成阶段 (Generation) ---

print("\n--- 生成阶段 ---")

# 加载一个用于生成答案的大型语言模型
# 这里使用一个轻量级的 distilgpt2 作为示例，你可以换成更强大的模型
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# 如果模型没有 pad_token，则设置一个
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 将检索到的上下文和用户问题拼接成一个完整的输入
prompt = f"根据以下上下文回答问题。问题：{question}\n上下文：{retrieved_context}\n回答："

# 对输入进行编码
inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

# 生成答案
# temperature 参数控制生成文本的随机性，较低的值会使结果更确定
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

# 解码生成的文本并打印
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 找到生成部分
start_index = response.rfind("回答：") + len("回答：")
final_answer = response[start_index:].strip()

print(f"生成的最终答案: {final_answer}")

LoRA

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# 1. 加载数据集和分词器（使用 GLUE SST-2）
dataset = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(preprocess, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_ds = tokenized_dataset["train"]
eval_ds = tokenized_dataset["validation"]

# 2. 加载模型并应用 LoRA
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,  # 低秩
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()  # 显示可训练参数（少量）

# 3. 设置训练参数并训练
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
)

trainer = Trainer(
    model=peft_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)

trainer.train()

# 4. 保存和简单推理
peft_model.save_pretrained("./lora-bert")

# 推理示例（加载合并后）
from peft import AutoPeftModelForSequenceClassification
merged_model = AutoPeftModelForSequenceClassification.from_pretrained("./lora-bert", num_labels=2)
merged_model = merged_model.merge_and_unload()

inputs = tokenizer("This is great!", return_tensors="pt")
outputs = merged_model(**inputs)
prediction = torch.argmax(outputs.logits, dim=-1).item()
print(f"Prediction: {prediction}")  # 0 或 1（负面/正面）